# 05 — Test-Year Target Grid Alignment (FIXED)

This replaces the old workflow that aligned **all years** to the first NDVI raster.

**Corrected logic**
- Training/validation rasters remain native and are NOT processed here.
- Only the independent spatial prediction year (2022) is resampled to one target grid.
- Raw source rasters are used directly; no "aligned → aligned again" chain.
- No nearest-value filling of large NoData regions.
- No edge clamping.
- Bilinear interpolation is used for continuous variables.
- Output NoData remains NoData.

In [ ]:
from pathlib import Path
import warnings

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists():
            return candidate
    raise FileNotFoundError(
        "Project root not found. Run this notebook from inside the repository."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"

for d in [INTERIM_DIR, PROCESSED_DIR, OUTPUT_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)

In [ ]:
import re, math
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling, transform_bounds
from rasterio.transform import from_origin
from rasterio.features import geometry_mask
import geopandas as gpd
from pyproj import CRS

TEST_YEAR = 2022
TARGET_RES_DEG = 0.005
DST_CRS = "EPSG:4326"
DST_NODATA = -9999.0

PRECIP_PRODUCTS = ["CCS","PDIR","GSMaP_MVK","CDR","CHIRPS","IMERG","GSMaP_Gauge_v7","ERA5"]
LAND_DYNAMIC = ["NDVI","LST_Day"]

def parse_ym(name):
    stem = Path(name).stem
    for pat in [r"(?<!\d)(20\d{2})[_-](0?[1-9]|1[0-2])(?!\d)",
                r"(?<!\d)(20\d{2})(0[1-9]|1[0-2])(?!\d)"]:
        m = re.search(pat, stem)
        if m:
            return int(m.group(1)), int(m.group(2))
    return None

def list_rasters(folder):
    return sorted([*folder.rglob("*.tif"), *folder.rglob("*.tiff")])

def monthly_map(folder):
    out = {}
    for p in list_rasters(folder):
        ym = parse_ym(p.name)
        if ym:
            if ym in out:
                raise ValueError(f"Duplicate month {ym} in {folder}")
            out[ym] = p
    return out

def source_crs_or_verified_wgs84(src, path):
    if src.crs is not None:
        return src.crs
    b = src.bounds
    geographic_bounds = (-180 <= b.left <= 180 and -180 <= b.right <= 180 and
                         -90 <= b.bottom <= 90 and -90 <= b.top <= 90)
    if geographic_bounds:
        warnings.warn(f"{path.name}: missing CRS but geographic-looking bounds; assuming EPSG:4326.")
        return CRS.from_epsg(4326)
    raise ValueError(f"{path}: CRS missing and bounds are not safely interpretable as lon/lat.")

In [ ]:
# Find study boundary and convert it to WGS84.
boundary_dir = RAW_DIR / "boundary"
boundary_candidates = [
    *boundary_dir.rglob("*.gpkg"),
    *boundary_dir.rglob("*.shp"),
    *boundary_dir.rglob("*.geojson"),
]
if not boundary_candidates:
    raise FileNotFoundError("No boundary vector found under data/raw/boundary")

boundary_path = boundary_candidates[0]
gdf = gpd.read_file(boundary_path)
if gdf.crs is None:
    raise ValueError("Study boundary has no CRS.")
gdf = gdf.to_crs(DST_CRS)
gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
if gdf.empty:
    raise ValueError("Boundary contains no valid geometry.")

minx, miny, maxx, maxy = gdf.total_bounds

# Snap outward to target grid.
left   = math.floor(minx / TARGET_RES_DEG) * TARGET_RES_DEG
right  = math.ceil (maxx / TARGET_RES_DEG) * TARGET_RES_DEG
bottom = math.floor(miny / TARGET_RES_DEG) * TARGET_RES_DEG
top    = math.ceil (maxy / TARGET_RES_DEG) * TARGET_RES_DEG

width = int(round((right-left)/TARGET_RES_DEG))
height = int(round((top-bottom)/TARGET_RES_DEG))
transform = from_origin(left, top, TARGET_RES_DEG, TARGET_RES_DEG)

inside_mask = geometry_mask(
    gdf.geometry,
    transform=transform,
    invert=True,
    out_shape=(height,width),
    all_touched=False,
)

print("Boundary:", boundary_path)
print("Target grid:", width, "x", height)
print("Bounds:", (left,bottom,right,top))
print("Grid spacing:", TARGET_RES_DEG, "degree")

In [ ]:
# Canonical static predictors.
def choose_static(folder_name, preferred_names):
    folder = RAW_DIR / "predictors" / folder_name
    files = list_rasters(folder)
    if not files:
        raise FileNotFoundError(f"No raster for {folder_name}")
    d = {p.name.lower():p for p in files}
    for n in preferred_names:
        if n.lower() in d:
            return d[n.lower()]
    clean = [p for p in files if not any(x in p.stem.lower() for x in ["clip","tmp","temp","aligned","resampl"])]
    if len(clean)==1:
        return clean[0]
    if len(files)==1:
        return files[0]
    raise ValueError(f"Ambiguous static predictor {folder_name}: {files}")

DEM_PATH = choose_static("DEM", ["Khulna_SRTM_DEM.tif","DEM.tif"])
DFS_PATH = choose_static("Distance_Sea", ["Distance_Sea.tif","distance_to_sea.tif"])

sources = {}
for product in PRECIP_PRODUCTS:
    mm = monthly_map(RAW_DIR/"precipitation"/product)
    for month in range(1,13):
        key = (product, month)
        if (TEST_YEAR,month) not in mm:
            raise FileNotFoundError(f"Missing {product} {TEST_YEAR}-{month:02d}")
        sources[key] = mm[(TEST_YEAR,month)]

for pred in LAND_DYNAMIC:
    mm = monthly_map(RAW_DIR/"predictors"/pred)
    for month in range(1,13):
        if (TEST_YEAR,month) not in mm:
            raise FileNotFoundError(f"Missing {pred} {TEST_YEAR}-{month:02d}")
        sources[(pred,month)] = mm[(TEST_YEAR,month)]

sources[("DEM",0)] = DEM_PATH
sources[("Distance_Sea",0)] = DFS_PATH
print("Sources prepared:", len(sources))

In [ ]:
aligned_root = PROCESSED_DIR / "test2022_target_grid"
aligned_root.mkdir(parents=True, exist_ok=True)

profile = {
    "driver":"GTiff", "height":height, "width":width, "count":1,
    "dtype":"float32", "crs":DST_CRS, "transform":transform,
    "nodata":DST_NODATA, "compress":"deflate", "predictor":2
}

qc_rows = []

def align_one(src_path, out_path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    dst = np.full((height,width), DST_NODATA, dtype="float32")
    with rasterio.open(src_path) as src:
        src_crs = source_crs_or_verified_wgs84(src, src_path)
        reproject(
            source=rasterio.band(src,1),
            destination=dst,
            src_transform=src.transform,
            src_crs=src_crs,
            src_nodata=src.nodata,
            dst_transform=transform,
            dst_crs=DST_CRS,
            dst_nodata=DST_NODATA,
            resampling=Resampling.bilinear,
            init_dest_nodata=True,
        )

    # Keep only valid study-area cells. Do not fill missing source coverage.
    dst[~inside_mask] = DST_NODATA
    valid = (dst != DST_NODATA) & np.isfinite(dst)
    with rasterio.open(out_path, "w", **profile) as out:
        out.write(dst,1)

    return {
        "source":str(src_path), "output":str(out_path),
        "valid_inside_pct":100*valid[inside_mask].sum()/inside_mask.sum(),
        "min":float(dst[valid].min()) if valid.any() else np.nan,
        "max":float(dst[valid].max()) if valid.any() else np.nan,
    }

for (name,month), src_path in sources.items():
    if month == 0:
        out_path = aligned_root / "static" / f"{name}.tif"
    else:
        out_path = aligned_root / name / f"{name}_{TEST_YEAR}_{month:02d}.tif"
    qc_rows.append(align_one(src_path, out_path))

qc = pd.DataFrame(qc_rows)
display(qc)
qc.to_csv(aligned_root / "alignment_qc.csv", index=False)

low = qc[qc.valid_inside_pct < 95]
if len(low):
    print("WARNING: target-grid coverage below 95% for some inputs. Do NOT fill these gaps blindly.")
    display(low)

print("Saved target-grid rasters to:", aligned_root)